# 05_train_clean — IC50 라벨링 + fingerprint 4종 학습·비교

**한 줄 요약:** IC50 ≤ 10000nM이면 **active(1)**, 아니면 **inactive(0)** 로 라벨을 붙이고, 4가지 fingerprint로 각각 모델을 학습해 **어느 지문이 좋은지** 5-fold 교차검증으로 비교한다.
**큰 흐름:** ① 준비 → ② 라벨링 → ③ 물질 단위 정리 → ④ 지문 4종 계산 → ⑤ 학습·비교

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 불러오기 + 상수
학습에 필요한 라이브러리(scikit-learn, LightGBM 포함)를 가져오고, 활성 임계값 등 상수를 정한다.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, average_precision_score)
from lightgbm import LGBMClassifier

SRC = "data/HSD17B13_IC50_merged.xlsx"
ACTIVE_MAX = 10000.0   # IC50 <= 10000nM => active
NBITS = 1024

🔎 **코드 뜯어보기 (셀 1)**
- `from sklearn.model_selection import StratifiedKFold, cross_val_predict, ...` : **scikit-learn**의 교차검증 도구. `StratifiedKFold`=클래스 비율 유지 분할, `cross_val_predict`=교차검증 예측.
- `from sklearn.metrics import (roc_auc_score, ...)` : 점수 계산 함수들.
- `from lightgbm import LGBMClassifier` : **LightGBM** 모델(빠른 부스팅 트리). `ACTIVE_MAX = 10000.0`=활성 임계값 상수.

### 셀 2 — 라벨링 규칙 + 데이터 읽고 라벨 붙이기
IC50/부등호로 active/inactive를 정하는 함수를 만들고, 3번째 시트를 읽어 라벨을 붙인다. 회색지대(10000~20000)도 확인.

In [ ]:
def make_label(ic50, rel):
    rel = str(rel).strip()
    if pd.isna(ic50):
        return np.nan
    if rel in ("<", "<="):          # 상한: value 이하가 확실 → value<=10000이면 active 확정
        return 1 if ic50 <= ACTIVE_MAX else 0
    if rel in (">", ">="):          # 하한: value 이상 → 항상 inactive(>10000 취지에 부합)
        return 0
    return 1 if ic50 <= ACTIVE_MAX else 0   # '=' 등

df = pd.read_excel(SRC, sheet_name="same_dedup_keepdiff")
df = df.dropna(subset=["canonical_smiles", "ic50_nM"]).copy()
df["label"] = [make_label(v, r) for v, r in zip(df["ic50_nM"], df["relation"])]
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print("=== 행(row) 기준 라벨 분포 ===")
print(df["label"].value_counts().rename({1: "active", 0: "inactive"}).to_string())
gray = df[(df["relation"].astype(str).str.strip() == "=") &
          (df["ic50_nM"] > 10000) & (df["ic50_nM"] < 20000)]
print(f"회색지대(10000~20000nM, '=') 행: {len(gray)} → inactive 처리됨")

🔎 **코드 뜯어보기 (셀 2)**
- `def make_label(ic50, rel):` : IC50 값과 부등호(rel)를 받아 1/0/빈값을 돌려주는 함수. `if rel in ("<","<="):`=부등호가 이 중 하나인지. `1 if 조건 else 0`=조건부 값.
- `[make_label(v, r) for v, r in zip(df["ic50_nM"], df["relation"])]` : **zip**=두 열을 짝지어 함께 반복. 각 (값,부등호)에 함수 적용해 라벨 리스트 생성.
- `.dropna(subset=["label"])` : 라벨 못 정한 행 제거. `.astype(int)`=정수로.
- `.value_counts().rename({1:"active",0:"inactive"})` : 개수를 세고 1/0을 글자로 바꿔 보기 좋게.
- `df[(조건A) & (조건B) & (조건C)]` : 여러 조건을 `&`로 결합해 회색지대 행만 고르기.

### 셀 3 — 물질(canonical) 단위로 정리
같은 분자가 학습·검증에 겹쳐 점수가 부풀지 않도록 물질 단위로 합친다(하나라도 active면 active).

In [ ]:
comp = (df.groupby("canonical_smiles")
          .agg(label=("label", "max"),
               n_rows=("label", "size"),
               contradictory=("label", lambda s: s.nunique() > 1))
          .reset_index())
print(f"\n=== 물질(canonical) 기준: {len(comp)}개 ===")
print(comp["label"].value_counts().rename({1: "active", 0: "inactive"}).to_string())
print(f"라벨이 충돌한 물질(측정마다 active/inactive 갈림): {int(comp['contradictory'].sum())}개")

🔎 **코드 뜯어보기 (셀 3)**
- `df.groupby("canonical_smiles").agg(label=("label","max"), n_rows=("label","size"), contradictory=("label", lambda s: s.nunique()>1))` : 물질별 집계. `"max"`=하나라도 1이면 1(active 우선), `"size"`=측정 수, **lambda**=즉석 함수로 '라벨이 갈렸는지'(nunique>1) 판정.

### 셀 4 — fingerprint 4종 계산
각 물질을 ECFP4·MACCS·RDKit·AtomPair 지문으로 변환해 모은다.

In [ ]:
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
gen_rdk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NBITS)
gen_ap = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NBITS)

def maccs_np(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps = {"ECFP4": [], "MACCS": [], "RDKit": [], "AtomPair": []}
y, keep = [], []
for smi, lab in zip(comp["canonical_smiles"], comp["label"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    fps["ECFP4"].append(gen_ecfp.GetFingerprintAsNumPy(mol))
    fps["RDKit"].append(gen_rdk.GetFingerprintAsNumPy(mol))
    fps["AtomPair"].append(gen_ap.GetFingerprintAsNumPy(mol))
    fps["MACCS"].append(maccs_np(mol))
    y.append(lab)
y = np.array(y)
print(f"\n학습 대상 물질: {len(y)}개 (active {int(y.sum())}, inactive {int((y==0).sum())})")

🔎 **코드 뜯어보기 (셀 4)** *(생성기·maccs_np는 04에서 설명)*
- `fps = {"ECFP4": [], ...}` : 지문 종류별 빈 리스트. `for smi, lab in zip(comp["canonical_smiles"], comp["label"])`=SMILES와 라벨을 함께 반복. `y.append(lab)`=정답 모으기. `np.array(y)`=리스트를 배열로.

### 셀 5 — 4종 지문으로 학습·비교 (5-fold CV)
지문마다 LightGBM 모델을 5-fold 교차검증으로 채점하고 최고를 고른다.

In [ ]:
print("\n" + "=" * 60)
print("Fingerprint별 5-fold 교차검증 결과 (LightGBM, class_weight=balanced)")
print("=" * 60)
print(f"{'Fingerprint':10s} {'ROC-AUC':>9s} {'PR-AUC':>9s} {'Recall(act)':>12s} {'Precision(act)':>15s}")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, mat in fps.items():
    X = np.vstack(mat)
    clf = LGBMClassifier(n_estimators=400, class_weight="balanced",
                         random_state=42, n_jobs=-1, verbosity=-1)
    proba = cross_val_predict(clf, X, y, cv=skf, method="predict_proba",
                              n_jobs=-1)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    pr = average_precision_score(y, proba)
    tn, fp_, fn, tp = confusion_matrix(y, pred).ravel()
    recall = tp / (tp + fn) if (tp + fn) else 0
    prec = tp / (tp + fp_) if (tp + fp_) else 0
    results[name] = auc
    print(f"{name:10s} {auc:9.3f} {pr:9.3f} {recall:12.3f} {prec:15.3f}")

best = max(results, key=results.get)
print(f"\n>>> 최고 fingerprint: {best} (ROC-AUC {results[best]:.3f})")

🔎 **코드 뜯어보기 (셀 5)**
- `for name, mat in fps.items():` : 지문 종류별 반복. `np.vstack(mat)`=배열들을 표로 쌓기.
- `LGBMClassifier(n_estimators=400, class_weight="balanced", ...)` : 모델. 괄호 안이 **하이퍼파라미터**(트리 400개 등, 성능 튜닝 대상). `verbosity=-1`=로그 끔.
- `cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]` : 5-fold 교차검증으로 '안 본 상태'의 active 확률.
- `confusion_matrix(y, pred).ravel()` : 맞고 틀림 표를 (tn,fp,fn,tp) 네 값으로 펴기 → Recall·Precision 계산.
- `max(results, key=results.get)` : 딕셔너리에서 **값(AUC)이 가장 큰 키(지문 이름)** 를 고르기.